In [ ]:
import pandas as pd

df_revenus = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',sep=';'
)

print(f"Shape : {df_revenus.shape}")
df_revenus.sample(5)

In [ ]:
list(df_revenus.columns)

signification des préfixes
[DISP]

👉 Revenu disponible des ménages.
C’est le revenu après impôts directs + prestations sociales.

Donc proche du pouvoir d’achat réel.

[DEC]

👉 Revenu déclaré / fiscal avant redistribution complète (selon source/table précise).
Souvent utilisé pour la structure fiscale.


Différence utile :

Pour analyser consommation / adoption VE :

👉 [DISP] est souvent plus pertinent que [DEC]

📘 Signification des variables principales
Identifiants géographiques
Nom géographique GMS : nom de la zone
Code géographique : code INSEE / code zone
Libellé géographique : nom officiel
👨‍👩‍👧 Taille / population fiscale
Nbre de ménages fiscaux : nombre de foyers fiscaux
Nbre de personnes dans les ménages fiscaux
Nbre d'unités de consommation

👉 utile pour normaliser les données

💶 Niveau de vie
1er quartile : 25% gagnent moins
Médiane : revenu central
3e quartile : 75% gagnent moins

👉 excellent indicateur de richesse locale

📊 Dispersion / inégalités
Écart interquartile = Q3 - Q1
Déciles D1 à D9
Rapport interdécile D9/D1
S80/20 : revenus des 20% les plus riches / 20% les plus pauvres
Indice de Gini

👉 mesure des inégalités

💼 Origine des revenus
Part des revenus d’activité
Part salaires
Part chômage
Part activités non salariées
Part pensions / retraites
Part patrimoine
Part prestations sociales
Part minima sociaux
Part logement
Part impôts

👉 structure socio-économique locale

Gini = vision globale
D9/D1 = contraste riches/pauvres lisible

On choisit donc de garder pour la modélisation :
['[DISP] Médiane (€)',
 '[DISP] Rapport interdécile 9ᵉ décile/1ᵉʳ decile',
 '[DISP] Iice de Gini',
 '[DISP] Nbre de ménages fiscaux',
 #population
 '[DISP] Nbre de personnes dans les ménages fiscaux'
 #zone dynamique économiquement
 '[DISP] Part des revenus d’activité (%)'
 #plus pauvre
 '[DISP] dont part des iemnités de chômage (%)'
 "[DISP] Part de l'ensemble des prestations sociales (%)"
 #plus riche
 '[DISP] dont part des revenus des activités non salariées (%)'
 '[DISP] Part des revenus du patrimoine et autres revenus (%)'
 #plus vieux
 '[DISP] Part des pensions, retraites et rentes (%)'
]

à créer :
Ratio actifs / retraités
part_activite / part_retraites
👉 dynamisme démographique.

3. Indice de précarité
chomage + minima_sociaux + prestations_sociales
(normalisé)

In [ ]:
var_selec = [
'[DISP] Médiane (€)',
 '[DISP] Rapport interdécile 9ᵉ décile/1ᵉʳ decile',
 '[DISP] Iice de Gini',
 '[DISP] Nbre de ménages fiscaux',
 #population
 '[DISP] Nbre de personnes dans les ménages fiscaux',
 #zone dynamique économiquement
 '[DISP] Part des revenus d’activité (%)',
 #plus pauvre
 '[DISP] dont part des iemnités de chômage (%)',
 "[DISP] Part de l'ensemble des prestations sociales (%)",
 #plus riche
 '[DISP] dont part des revenus des activités non salariées (%)',
 '[DISP] Part des revenus du patrimoine et autres revenus (%)',
 #plus vieux
 '[DISP] Part des pensions, retraites et rentes (%)'
]

df_filtre = df_revenus[var_selec]
df_filtre.sample(5)

In [ ]:
df_filtre.dtypes

In [ ]:
# ==========================================================
# ETUDE DES CORRELATIONS - Variables socio-économiques
# DataFrame supposé : df
# ==========================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ----------------------------------------------------------
# 2. Création dataframe numérique propre
# ----------------------------------------------------------

df_corr = df_filtre.copy()

# suppression lignes vides
df_corr = df_corr.dropna()

# ----------------------------------------------------------
# 3. Matrice de corrélation Spearman
# ----------------------------------------------------------

corr_matrix = df_corr.corr(method='spearman')

# ----------------------------------------------------------
# 4. Heatmap lisible
# ----------------------------------------------------------

plt.figure(figsize=(14,10))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt=".2f",
    linewidths=0.5
)

plt.title("Corrélations entre variables socio-économiques", fontsize=14)
plt.xticks(rotation=75, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# 5. Corrélations fortes uniquement
# ----------------------------------------------------------

print("\n===== CORRELATIONS FORTES (>|0.70|) =====")

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        val = corr_matrix.iloc[i, j]
        if abs(val) >= 0.70:
            print(
                f"{corr_matrix.columns[i]}  <-->  "
                f"{corr_matrix.columns[j]} : {val:.2f}"
            )


Finalement on va garder :

var_selec = [
'[DISP] Médiane (€)',
 '[DISP] Iice de Gini',
 #garder uniquement l'un des deux
 '[DISP] Nbre de ménages fiscaux',
 '[DISP] Nbre de personnes dans les ménages fiscaux',
 #zone dynamique économiquement (potentiellement plus jeune aussi)
 '[DISP] Part des revenus d’activité (%)',
 #plus riche
 '[DISP] dont part des revenus des activités non salariées (%)',
 '[DISP] Part des revenus du patrimoine et autres revenus (%)',
]